# 06 — State representation research: the Fatimah experiment

This notebook turns the Elder Mist toy into the research design you've been trying to parse.

The question is **not**:

> How much telemetry can we capture?

It is:

> Given available raw information, what representation lets a fixed decision-making system act best?

We will keep the environment and learning algorithm fixed and change only the representation.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo))

from eldermist import ElderMistReplicaEnv
from eldermist.qlearning import train_q, q_policy


## 1. Candidate representations

Our simulator's true state contains lots of facts.

We deliberately expose different subsets:

```text
FULL
actor + party HP + boss HP + Sword + Sword HP
+ charged move + timer + locks + combo + MP + shields

NO_SWORD
remove Sword state

NO_LOCKS
remove charged-move/lock information

MINIMAL
actor + party HP + boss HP
```

This is an **ablation**: remove one information source and measure what breaks.


In [ ]:
representations = ["full", "no_sword", "no_locks", "minimal"]

for rep in representations:
    env = ElderMistReplicaEnv(representation=rep)
    env.reset(seed=1)
    print(rep, "->", env.compact_state())


## 2. Experimental hygiene

We will:

1. use the same Q-learning algorithm,
2. train each representation with the same number of episodes,
3. evaluate on **different held-out random seeds**,
4. collect several metrics.

That prevents us from saying a representation is “better” because we happened to show it easier fights.


In [ ]:
def train_and_eval(rep, train_episodes=8000, eval_episodes=250):
    q, train_rows = train_q(
        lambda: ElderMistReplicaEnv(representation=rep, reward_mode="shaped"),
        episodes=train_episodes,
        seed=100,
    )

    eval_rows = []
    for seed in range(50_000, 50_000 + eval_episodes):
        env = ElderMistReplicaEnv(representation=rep, reward_mode="shaped")
        obs, _ = env.reset(seed=seed)
        total = 0.0

        while True:
            action = q_policy(q, env)
            obs, reward, term, trunc, _ = env.step(action)
            total += reward
            if term or trunc:
                break

        eval_rows.append({
            "win": int(env.state.boss_hp <= 0),
            "turns": env.state.turn,
            "counter_slashes": env.state.counter_slashes,
            "specials_cancelled": env.state.specials_cancelled,
            "return": total,
        })

    e = pd.DataFrame(eval_rows)
    return q, pd.DataFrame(train_rows), {
        "representation": rep,
        "win_rate": e["win"].mean(),
        "avg_turns": e["turns"].mean(),
        "avg_counter_slashes": e["counter_slashes"].mean(),
        "avg_specials_cancelled": e["specials_cancelled"].mean(),
        "avg_return": e["return"].mean(),
    }

results = []
models = {}

for rep in representations:
    print("training", rep)
    q, train_df, metrics = train_and_eval(rep)
    models[rep] = q
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(results_df["representation"], results_df["win_rate"])
plt.ylim(0,1)
plt.ylabel("Held-out win rate")
plt.title("Downstream performance by state representation")
plt.show()


## 3. What does “best” mean?

Not necessarily highest win rate at any cost.

For computer agents you might care about:

```text
maximize:
    task success

minimize:
    tokens
    latency
    number of tool calls
    invalid actions
    policy violations
```

So we can add a **representation cost**.

Below is an illustrative cost: number of fields in our compact representation.


In [ ]:
field_cost = {
    "full": 13,
    "no_sword": 11,
    "no_locks": 10,
    "minimal": 4,
}

results_df["representation_fields"] = results_df["representation"].map(field_cost)
results_df[[
    "representation",
    "win_rate",
    "avg_turns",
    "avg_counter_slashes",
    "representation_fields",
]]


In [ ]:
plt.figure(figsize=(7,5))

for _, row in results_df.iterrows():
    plt.scatter(row["representation_fields"], row["win_rate"], s=80)
    plt.text(
        row["representation_fields"] + 0.1,
        row["win_rate"],
        row["representation"]
    )

plt.xlabel("Representation size (field count)")
plt.ylabel("Held-out win rate")
plt.ylim(0,1)
plt.title("Performance vs representation cost")
plt.show()


## 4. Pareto-optimal representation

A candidate is **dominated** if another candidate is:

- at least as successful,
- no more expensive,
- and strictly better on at least one.

The non-dominated candidates form the Pareto frontier.


In [ ]:
def pareto_frontier(df):
    keep = []
    for i, a in df.iterrows():
        dominated = False
        for j, b in df.iterrows():
            if i == j:
                continue
            no_worse = (
                b["win_rate"] >= a["win_rate"]
                and b["representation_fields"] <= a["representation_fields"]
            )
            strictly_better = (
                b["win_rate"] > a["win_rate"]
                or b["representation_fields"] < a["representation_fields"]
            )
            if no_worse and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(a["representation"])

    return df[df["representation"].isin(keep)]

pareto_frontier(results_df)


## 5. Translate this directly to screenshot + accessibility tree

Candidate experiment:

```text
A = screenshot
B = accessibility tree
C = screenshot + tree
D = screenshot + tree + recent history
E = LLM-produced structured task state
F = learned embedding of all sources
```

Keep fixed:

```text
same tasks
same underlying model
same tool/action space
same reward/evaluator
```

Measure:

```text
task success
steps
tokens
latency
invalid tool calls
policy violations
```

Then compare on held-out tasks.


## 6. The questions to ask Fatimah

Bring these into the brainstorm:

1. **What are the raw sources?**
   Screenshot, accessibility tree, cursor, keyboard, history, app APIs?

2. **What exactly is the action space?**
   Raw mouse/keyboard actions, semantic UI actions, tool calls, or high-level options?

3. **What is an episode?**
   One benchmark ticket? One computer task? One user objective?

4. **What is the environment transition?**
   Is it a simulator, browser/computer sandbox, logged offline trajectory, or live system?

5. **What is reward?**
   Binary completion, graded assertions, human correction, latency/cost, policy compliance?

6. **What is being trained when we say “train the harness”?**
   State encoder? LLM? LoRA? retrieval? memory selector? planner? tool policy?

7. **How do we compare state representations fairly?**
   Same model, same tasks, same training budget, held-out evaluation.

8. **Are we optimizing imitation or outcomes?**
   “Would this employee do it?” and “does this solve the task?” are different objectives.

9. **Where is uncertainty represented?**
   If important facts are hidden, do we use history, recurrent state, or an explicit belief?

10. **What is the smallest sufficient representation?**
    Which information can we remove without hurting decisions?


## 7. Why τ² is useful

τ² is useful because it gives you a controlled loop where:

```text
state / observations
→ tool or dialogue action
→ simulated world changes
→ new observation
→ machine-checkable outcome
```

That makes it a convenient place to test **representation ablations** before dealing with the messiness of real employee telemetry.

The key point:

> τ² is not important because telecom is special. It is useful because the environment, actions, transitions, and success conditions are controllable.


### Sentence to memorize

```text
State-representation research asks:
"What information must the policy retain so that different future consequences
and good actions remain distinguishable?"
```
